# Week 9 — Handwritten Digit Classifier with PyTorch

**Theme:** PyTorch basics — building and training a neural network from scratch

So far, scikit-learn has hidden the training loop from us (`.fit()` did
everything). This week we open that black box: PyTorch makes you write the
loop yourself, which is exactly what lets you build custom architectures later
(CNNs in Week 10, GNNs in Week 11, RNNs in Week 13, and today's LLMs are all
built the same way, just bigger).

**Dataset:** MNIST — 70,000 images of handwritten digits (0-9), 28x28
grayscale pixels each. `torchvision` downloads it automatically the first time.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## 1. Tensors, in 60 seconds

A PyTorch `Tensor` is like a NumPy array that can also track gradients and run
on a GPU. That's really all you need to know to get started.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x ** 2).sum()   # y = x0^2 + x1^2 + x2^2
y.backward()          # compute dy/dx automatically
print("x       :", x)
print("y       :", y.item())
print("dy/dx   :", x.grad)  # should be 2*x = [2, 4, 6]

## 2. Load MNIST

`transforms.ToTensor()` converts each image to a tensor with pixel values
scaled to [0, 1].

In [ ]:
transform = transforms.ToTensor()

train_data = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False)

print(f"Training images: {len(train_data)}, test images: {len(test_data)}")

In [ ]:
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(8, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i][0], cmap="gray_r")
    ax.set_title(str(labels[i].item()))
    ax.axis("off")
plt.suptitle("A batch of MNIST digits")
plt.show()

## 3. Define the network

A simple feedforward network: flatten the 28x28 image into 784 numbers, pass
through one hidden layer with a ReLU activation, then a 10-way output layer
(one score per digit).

In [ ]:
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)   # flatten (batch, 1, 28, 28) -> (batch, 784)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)             # raw scores ("logits"), one per digit
        return x

model = DigitClassifier().to(device)
print(model)
print("Total trainable parameters:", sum(p.numel() for p in model.parameters()))

## 4. The training loop

Every PyTorch training loop follows the same four steps for each batch:
1. **Forward pass** — run the batch through the model
2. **Loss** — how wrong were the predictions?
3. **Backward pass** — compute gradients of the loss w.r.t. every parameter
4. **Optimizer step** — nudge every parameter to reduce the loss

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

n_epochs = 3
history = {"train_loss": [], "test_acc": []}

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)          # 1. forward pass
        loss = criterion(outputs, labels)  # 2. loss
        loss.backward()                  # 3. backward pass (gradients)
        optimizer.step()                 # 4. optimizer step (update weights)

        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_data)
    test_acc = evaluate(model, test_loader)
    history["train_loss"].append(train_loss)
    history["test_acc"].append(test_acc)
    print(f"Epoch {epoch+1}/{n_epochs}  train_loss={train_loss:.4f}  test_acc={test_acc:.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history["train_loss"], marker="o")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")

axes[1].plot(history["test_acc"], marker="o", color="seagreen")
axes[1].set_title("Test Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 5. Look at some mistakes

A trained model is rarely perfect — looking at *which* digits it confuses is
often more informative than the accuracy number alone.

In [ ]:
model.eval()
wrong_images, wrong_true, wrong_pred = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images_d, labels_d = images.to(device), labels.to(device)
        preds = model(images_d).argmax(dim=1)
        mistakes = (preds != labels_d).cpu()
        if mistakes.any():
            wrong_images.extend(images[mistakes][:10 - len(wrong_images)])
            wrong_true.extend(labels[mistakes][:10 - len(wrong_true)])
            wrong_pred.extend(preds.cpu()[mistakes][:10 - len(wrong_pred)])
        if len(wrong_images) >= 10:
            break

fig, axes = plt.subplots(2, 5, figsize=(9, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(wrong_images[i][0], cmap="gray_r")
    ax.set_title(f"true={wrong_true[i].item()}, pred={wrong_pred[i].item()}")
    ax.axis("off")
plt.suptitle("Misclassified Digits")
plt.tight_layout()
plt.show()

## Try it yourself

1. **Add a second hidden layer.** Insert `nn.Linear(128, 64)` +
   `F.relu` before `fc2`, and change `fc2` to `nn.Linear(64, 10)` — does
   accuracy improve?
2. **Train longer.** Bump `n_epochs` to 5 or 10 — where does accuracy start
   to plateau?
3. **Change the learning rate.** Try `lr=0.1` and `lr=0.0001` — what happens
   to the loss curve at each extreme?
4. **Count the parameters.** Work out by hand why the model has
   `28*28*128 + 128 + 128*10 + 10` parameters, and confirm it matches the
   printed total above.